In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [12]:
general_info = pd.read_csv(
    'Hospital_General_Information.csv',
    dtype={'Facility ID': str},
    low_memory=False
)

print(f"Raw shape: {general_info.shape}")
general_info.head()

Raw shape: (5426, 38)


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Hospital Type,Hospital Ownership,...,Count of READM Measures Better,Count of READM Measures No Different,Count of READM Measures Worse,READM Group Footnote,Pt Exp Group Measure Count,Count of Facility Pt Exp Measures,Pt Exp Group Footnote,TE Group Measure Count,Count of Facility TE Measures,TE Group Footnote
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,0,11,0,NaN,8,8,NaN,12,11,NaN
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,0,8,1,NaN,8,8,NaN,12,12,NaN
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,0,8,1,NaN,8,8,NaN,12,10,NaN
3,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,Acute Care Hospitals,Voluntary non-profit - Private,...,0,7,0,NaN,8,8,NaN,12,7,NaN
4,010008,CRENSHAW COMMUNITY HOSPITAL,101 HOSPITAL CIRCLE,LUVERNE,AL,36049,CRENSHAW,(334) 335-3374,Acute Care Hospitals,Proprietary,...,0,2,0,NaN,8,Not Available,5.0,12,6,NaN


In [15]:
print("--- Column names ---")
print(general_info.columns.tolist())

print("\n--- Dtypes ---")
print(general_info.dtypes)

print("\n--- Nulls (raw, before cleaning) ---")
print(general_info.isnull().sum().sort_values(ascending=False))

--- Column names ---
['Facility ID', 'Facility Name', 'Address', 'City/Town', 'State', 'ZIP Code', 'County/Parish', 'Telephone Number', 'Hospital Type', 'Hospital Ownership', 'Emergency Services', 'Meets criteria for birthing friendly designation', 'Hospital overall rating', 'Hospital overall rating footnote', 'MORT Group Measure Count', 'Count of Facility MORT Measures', 'Count of MORT Measures Better', 'Count of MORT Measures No Different', 'Count of MORT Measures Worse', 'MORT Group Footnote', 'Safety Group Measure Count', 'Count of Facility Safety Measures', 'Count of Safety Measures Better', 'Count of Safety Measures No Different', 'Count of Safety Measures Worse', 'Safety Group Footnote', 'READM Group Measure Count', 'Count of Facility READM Measures', 'Count of READM Measures Better', 'Count of READM Measures No Different', 'Count of READM Measures Worse', 'READM Group Footnote', 'Pt Exp Group Measure Count', 'Count of Facility Pt Exp Measures', 'Pt Exp Group Footnote', 'TE Grou

In [16]:
general_info.columns = (
    general_info.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('/', '_')
    .str.replace(r'[^\w]', '_', regex=True)
)

print("Columns after renaming:")
print(general_info.columns.tolist())



Columns after renaming:
['facility_id', 'facility_name', 'address', 'city_town', 'state', 'zip_code', 'county_parish', 'telephone_number', 'hospital_type', 'hospital_ownership', 'emergency_services', 'meets_criteria_for_birthing_friendly_designation', 'hospital_overall_rating', 'hospital_overall_rating_footnote', 'mort_group_measure_count', 'count_of_facility_mort_measures', 'count_of_mort_measures_better', 'count_of_mort_measures_no_different', 'count_of_mort_measures_worse', 'mort_group_footnote', 'safety_group_measure_count', 'count_of_facility_safety_measures', 'count_of_safety_measures_better', 'count_of_safety_measures_no_different', 'count_of_safety_measures_worse', 'safety_group_footnote', 'readm_group_measure_count', 'count_of_facility_readm_measures', 'count_of_readm_measures_better', 'count_of_readm_measures_no_different', 'count_of_readm_measures_worse', 'readm_group_footnote', 'pt_exp_group_measure_count', 'count_of_facility_pt_exp_measures', 'pt_exp_group_footnote', 'te_g

In [17]:


CMS_NULLS = [
    'Not Available',
    'Not Applicable',
    'Too Few to Report',
    'N/A'
]

general_info = general_info.replace(CMS_NULLS, np.nan)

print("county_parish sample after replace:")
print(general_info['county_parish'].value_counts(dropna=False).head(10))

county_parish sample after replace:
county_parish
LOS ANGELES    88
COOK           59
JEFFERSON      59
MONTGOMERY     55
MARICOPA       53
WASHINGTON     52
HARRIS         49
ORANGE         46
FRANKLIN       43
CLARK          39
Name: count, dtype: int64


In [18]:
general_info['facility_id'] = (
    general_info['facility_id']
    .str.strip()
    .str.zfill(6)
)

general_info['zip_code'] = (
    general_info['zip_code']
    .astype(str)
    .str.strip()
    .str.zfill(5)
)

print("facility_id sample:", general_info['facility_id'].head().tolist())
print("zip_code sample   :", general_info['zip_code'].head().tolist())

facility_id sample: ['010001', '010005', '010006', '010007', '010008']
zip_code sample   : ['36301', '35957', '35630', '36467', '36049']


In [19]:
cat_cols = [
    'hospital_type',
    'hospital_ownership',
    'state',
    'emergency_services'
]

for col in cat_cols:
    if col in general_info.columns:
        general_info[col] = general_info[col].str.strip()

print("Whitespace stripped from categorical columns")

Whitespace stripped from categorical columns


In [20]:
numeric_cols = [
    col for col in general_info.columns
    if any(kw in col for kw in ['count', 'rating', 'score'])
]

for col in numeric_cols:
    general_info[col] = pd.to_numeric(general_info[col], errors='coerce')

print(f"Cast {len(numeric_cols)} columns to numeric:")
print(numeric_cols)

Cast 22 columns to numeric:
['county_parish', 'hospital_overall_rating', 'hospital_overall_rating_footnote', 'mort_group_measure_count', 'count_of_facility_mort_measures', 'count_of_mort_measures_better', 'count_of_mort_measures_no_different', 'count_of_mort_measures_worse', 'safety_group_measure_count', 'count_of_facility_safety_measures', 'count_of_safety_measures_better', 'count_of_safety_measures_no_different', 'count_of_safety_measures_worse', 'readm_group_measure_count', 'count_of_facility_readm_measures', 'count_of_readm_measures_better', 'count_of_readm_measures_no_different', 'count_of_readm_measures_worse', 'pt_exp_group_measure_count', 'count_of_facility_pt_exp_measures', 'te_group_measure_count', 'count_of_facility_te_measures']


In [56]:
dupes = general_info.duplicated(subset=['facility_id']).sum()
print(f"Duplicate facility_ids: {dupes}")


Duplicate facility_ids: 0


In [21]:
print(f"Final shape      : {general_info.shape}")
print(f"Unique hospitals : {general_info['facility_id'].nunique()}")

print("\n--- Nulls after cleaning (top 20) ---")
print(general_info.isnull().sum().sort_values(ascending=False).head(20))

print("\n--- Hospital types ---")
print(general_info['hospital_type'].value_counts(dropna=False))

print("\n--- Emergency services ---")
print(general_info['emergency_services'].value_counts(dropna=False))

print("\n--- Hospital ownership ---")
print(general_info['hospital_ownership'].value_counts(dropna=False))

print("\n--- county_parish check (should NOT be all NaN) ---")
print(general_info['county_parish'].value_counts(dropna=False).head(10))

Final shape      : (5426, 38)
Unique hospitals : 5426

--- Nulls after cleaning (top 20) ---
county_parish                                       5426
te_group_footnote                                   4486
readm_group_footnote                                4266
mort_group_footnote                                 3639
safety_group_footnote                               3347
meets_criteria_for_birthing_friendly_designation    3161
pt_exp_group_footnote                               3151
hospital_overall_rating_footnote                    2856
hospital_overall_rating                             2560
count_of_facility_pt_exp_measures                   2275
count_of_facility_safety_measures                   2073
count_of_safety_measures_worse                      2073
count_of_safety_measures_no_different               2073
count_of_safety_measures_better                     2073
count_of_mort_measures_no_different                 1786
count_of_mort_measures_worse                        

In [58]:
general_info.head()

,facility_id,facility_name,address,city_town,state,zip_code,county_parish,telephone_number,hospital_type,hospital_ownership,...,count_of_readm_measures_better,count_of_readm_measures_no_different,count_of_readm_measures_worse,readm_group_footnote,pt_exp_group_measure_count,count_of_facility_pt_exp_measures,pt_exp_group_footnote,te_group_measure_count,count_of_facility_te_measures,te_group_footnote
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,NaN,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,0.0,11.0,0.0,NaN,8.0,8.0,NaN,12.0,11.0,NaN
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,NaN,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,0.0,8.0,1.0,NaN,8.0,8.0,NaN,12.0,12.0,NaN
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,NaN,(256) 768-8400,Acute Care Hospitals,Proprietary,...,0.0,8.0,1.0,NaN,8.0,8.0,NaN,12.0,10.0,NaN
3,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,NaN,(334) 493-3541,Acute Care Hospitals,Voluntary non-profit - Private,...,0.0,7.0,0.0,NaN,8.0,8.0,NaN,12.0,7.0,NaN
4,010008,CRENSHAW COMMUNITY HOSPITAL,101 HOSPITAL CIRCLE,LUVERNE,AL,36049,NaN,(334) 335-3374,Acute Care Hospitals,Proprietary,...,0.0,2.0,0.0,NaN,8.0,NaN,5.0,12.0,6.0,NaN


In [22]:
general_info.to_csv('general_info_cleaned.csv', index=False)
print("Saved")

Saved
